In [1]:

from pathlib import Path

folder = Path.home() / "Documents" / "EngiLearn"

print("Folder exists:", folder.exists())
print("Folder path:", folder)
print("Folder contents:", list(folder.iterdir()) if folder.exists() else "Folder not found")

Folder exists: True
Folder path: C:\Users\Chetana\Documents\EngiLearn
Folder contents: [WindowsPath('C:/Users/Chetana/Documents/EngiLearn/.git'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/.gitignore'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/cleaned_text.txt'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/EngiLearn_Project.ipynb'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/extracted_figures'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/extracted_text.txt'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/learning_material.json'), WindowsPath('C:/Users/Chetana/Documents/EngiLearn/sample_engineering.pdf.pdf')]


In [2]:

from pathlib import Path
import fitz

pdf_path = Path.home() / "Documents" / "EngiLearn" / "sample_engineering.pdf"

if pdf_path.exists():
    try:
        doc = fitz.open(pdf_path)
        print("PDF opened successfully!")
        print("Total pages:", len(doc))
    except Exception as e:
        print("Could not open as PDF:", e)
else:
    print("File not found. Check the filename.")

File not found. Check the filename.


In [3]:

from pathlib import Path
import pymupdf

folder = Path.home() / "Documents" / "EngiLearn"

pdf_files = list(folder.glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for file in pdf_files:
    print("Filename:", file.name)
    try:
        doc = pymupdf.open(file)
        print("PDF opened successfully!")
        print("Total pages:", len(doc))
    except Exception as e:
        print("Could not open this file:", e)


PDF files found: 1
Filename: sample_engineering.pdf.pdf
PDF opened successfully!
Total pages: 81


In [4]:

import pymupdf
from pathlib import Path

pdf_path = Path.home() / "Documents" / "EngiLearn" / "sample_engineering.pdf.pdf"

doc = pymupdf.open(pdf_path)

print("Total pages:", len(doc))

for page_number in range(min(3, len(doc))):
    page = doc[page_number]
    text = page.get_text()

    print(f"\n--- Page {page_number + 1} ---")
    print(text[:2000])


Total pages: 81

--- Page 1 ---
Lecture 1: Introduction to Deep Learning
6.S898 Deep Learning
Fall 2022
Speaker: Sara Beery
https://phillipi.github.io/6.s898
© Krizhevsky, Sutskever, and Hinton. All rights reserved. This 
content is excluded from our Creative Commons license. For more 
information, see https://ocw.mit.edu/help/faq-fair-use/
1


--- Page 2 ---
What is “deep learning”?
1. Neural nets: A class of machine learning architectures that
use stacks of linear transformations interleaved with 
pointwise nonlinearities 
2. Differentiable programming: A programming paradigm
where parameterize parts of the program and let gradient-
based optimization tune the parameters
2


--- Page 3 ---
Course philosophy
• Breakthroughs in deep learning have been driven by a mixture of theory
and practice, and both dimensions are vital for future progress in the field
• This course provides:
• Theoretical grounding in important deep learning building blocks
• Practice implementing, understanding, 

In [5]:

import pymupdf
from pathlib import Path
from PIL import Image
from io import BytesIO

# PDF location
folder = Path.home() / "Documents" / "EngiLearn"
pdf_path = folder / "sample_engineering.pdf.pdf"

# Folder to save extracted images
output_folder = folder / "extracted_figures"
output_folder.mkdir(parents=True, exist_ok=True)

doc = pymupdf.open(pdf_path)
image_count = 0

for page_number, page in enumerate(doc, start=1):
    for image_index, image_info in enumerate(page.get_images(full=True), start=1):
        try:
            image_data = doc.extract_image(image_info[0])
            image = Image.open(
                BytesIO(image_data["image"])
            ).convert("RGB")

            image_path = output_folder / (
                f"page_{page_number}_image_{image_index}.png"
            )
            image.save(image_path)
            image_count += 1

        except Exception as error:
            print(f"Skipped image on page {page_number}: {error}")

print("Image extraction complete!")
print("Images saved:", image_count)
print("Saved location:", output_folder)


Image extraction complete!
Images saved: 1643
Saved location: C:\Users\Chetana\Documents\EngiLearn\extracted_figures


In [6]:

import pymupdf
from pathlib import Path

folder = Path.home() / "Documents" / "EngiLearn"
pdf_path = folder / "sample_engineering.pdf.pdf"

doc = pymupdf.open(pdf_path)

all_text = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text().strip()

    if text:
        all_text.append(
            f"\n--- Page {page_number} ---\n{text}"
        )

full_text = "\n".join(all_text)

text_path = folder / "extracted_text.txt"
text_path.write_text(full_text, encoding="utf-8")

print("Text extraction complete!")
print("Pages processed:", len(doc))
print("Characters extracted:", len(full_text))
print("Text saved to:", text_path)

print("\nPreview:\n")
print(full_text[:2000])


Text extraction complete!
Pages processed: 81
Characters extracted: 19297
Text saved to: C:\Users\Chetana\Documents\EngiLearn\extracted_text.txt

Preview:


--- Page 1 ---
Lecture 1: Introduction to Deep Learning
6.S898 Deep Learning
Fall 2022
Speaker: Sara Beery
https://phillipi.github.io/6.s898
© Krizhevsky, Sutskever, and Hinton. All rights reserved. This 
content is excluded from our Creative Commons license. For more 
information, see https://ocw.mit.edu/help/faq-fair-use/
1

--- Page 2 ---
What is “deep learning”?
1. Neural nets: A class of machine learning architectures that
use stacks of linear transformations interleaved with 
pointwise nonlinearities 
2. Differentiable programming: A programming paradigm
where parameterize parts of the program and let gradient-
based optimization tune the parameters
2

--- Page 3 ---
Course philosophy
• Breakthroughs in deep learning have been driven by a mixture of theory
and practice, and both dimensions are vital for future progress in the

In [7]:
%pip install scikit-learn


In [8]:

from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

folder = Path.home() / "Documents" / "EngiLearn"
text_path = folder / "extracted_text.txt"

text = text_path.read_text(encoding="utf-8")

# Split the document into sentences
sentences = re.split(r'(?<=[.!?])\s+', text)
sentences = [
    s.strip().replace("\n", " ")
    for s in sentences
    if len(s.strip()) > 30
]

# Remove duplicate sentences
sentences = list(dict.fromkeys(sentences))

if len(sentences) < 2:
    print("Not enough sentences to summarize.")
else:
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(sentences)

    similarity = cosine_similarity(
        matrix, matrix
    ).sum(axis=1)

    top_indices = sorted(
        similarity.argsort()[-8:],
        reverse=True
    )

    summary = "\n".join(
        f"- {sentences[i]}" for i in top_indices
    )

    print("DOCUMENT SUMMARY\n")
    print(summary)


DOCUMENT SUMMARY

- This  content is excluded from our Creative  Commons license.
- This content is excluded from our  Creative Commons license.
- This content is excluded  from our Creative Commons license.
- This content is  excluded from our Creative Commons license.
- This content is excluded from our Creative  Commons license.
- For more  information, see https://ocw.mit.edu/help/faq-fair-use/ 8  --- Page 9 --- Perceptrons, 1958 © source unknown.
- For more  information, see https://ocw.mit.edu/help/faq-fair-use/ 1  --- Page 2 --- What is “deep learning”?
- This  content is excluded from our Creative Commons license.


In [9]:

from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

folder = Path.home() / "Documents" / "EngiLearn"
text_path = folder / "extracted_text.txt"
text = text_path.read_text(encoding="utf-8")

# Remove repeated copyright notices and URLs
text = re.sub(
    r'This\s+content\s+is\s+excluded\s+from\s+our\s+Creative\s+Commons\s+license\.?',
    ' ', text, flags=re.IGNORECASE
)
text = re.sub(r'https?://\S+', ' ', text)

# Remove page markers
text = re.sub(r'--- Page \d+ ---', ' ', text)

# Split text into sentences
sentences = re.split(r'(?<=[.!?])\s+', text)

clean_sentences = []
for sentence in sentences:
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    sentence = re.sub(r'^\d+\s+', '', sentence)

    if len(sentence.split()) >= 5:
        clean_sentences.append(sentence)

# Remove duplicate sentences
clean_sentences = list(dict.fromkeys(clean_sentences))

# Score sentences using TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)
matrix = vectorizer.fit_transform(clean_sentences)
scores = cosine_similarity(matrix, matrix).sum(axis=1)

# Select the highest-scoring sentences in document order
top_indices = sorted(
    scores.argsort()[-8:],
    key=lambda i: i
)

print("IMPROVED DOCUMENT SUMMARY\n")
for index in top_indices:
    print("-", clean_sentences[index])


IMPROVED DOCUMENT SUMMARY

- time enthusiasm Perceptrons, 1958 Minsky and Papert, 1972 PDP book, 1986 AI winter, 2000 18 Krizhevsky, Sutskever, and Hinton, NeurIPS 2012 “Alexnet” © Krizhevsky, Sutskever, and Hinton.
- For more information, see 54 Last layer … … dolphin cat grizzly bear angel ﬁsh chameleon iguana elephant clown ﬁsh “clown ﬁsh” argmax Classiﬁer layer 55 “clown ﬁsh” Loss error Network output … … dolphin cat grizzly bear angel ﬁsh chameleon iguana elephant clown ﬁsh Ground truth label Loss function 56 “clown ﬁsh” Loss small Network output … … dolphin cat grizzly bear angel ﬁsh chameleon iguana elephant clown ﬁsh Ground truth label Loss function 57 “grizzly bear” Loss large Network output … … dolphin cat grizzly bear angel ﬁsh chameleon iguana elephant clown ﬁsh Ground truth label Loss function 58 Network output … dolphin cat grizzly bear angel ﬁsh chameleon clown ﬁsh Ground truth label … iguana elephant ˆy y H (y , ˆy ) = − K k =1 yk log ˆyk Probability of the observed dat

In [10]:

from pathlib import Path
import re
import pymupdf

folder = Path.home() / "Documents" / "EngiLearn"
pdf_path = folder / "sample_engineering.pdf.pdf"

doc = pymupdf.open(pdf_path)
clean_pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text("text")

    # Remove URLs and copyright notices
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(
        r'This content is excluded.*?license\.?',
        '', text, flags=re.IGNORECASE | re.DOTALL
    )

    # Keep readable text lines
    lines = []
    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()

        if not line:
            continue
        if len(line) < 25:
            continue
        if re.fullmatch(r'[\d\s.,()\-]+', line):
            continue

        lines.append(line)

    if lines:
        clean_pages.append(
            f"PAGE {page_number}\n" + "\n".join(lines)
        )

clean_text = "\n\n".join(clean_pages)

output_path = folder / "cleaned_text.txt"
output_path.write_text(clean_text, encoding="utf-8")

print("Cleaned text saved to:", output_path)
print("\nPREVIEW:\n")
print(clean_text[:4000])


Cleaned text saved to: C:\Users\Chetana\Documents\EngiLearn\cleaned_text.txt

PREVIEW:

PAGE 1
Lecture 1: Introduction to Deep Learning
© Krizhevsky, Sutskever, and Hinton. All rights reserved. This
content is excluded from our Creative Commons license. For more

PAGE 2
1. Neural nets: A class of machine learning architectures that
use stacks of linear transformations interleaved with
2. Differentiable programming: A programming paradigm
where parameterize parts of the program and let gradient-
based optimization tune the parameters

PAGE 3
• Breakthroughs in deep learning have been driven by a mixture of theory
and practice, and both dimensions are vital for future progress in the field
• Theoretical grounding in important deep learning building blocks
• Practice implementing, understanding, and using those blocks

PAGE 4
•Our policy for using ChatGPT and other AI assistants is identical to our policy for using
•This is a deep learning class and you should try out all the latest AI as

In [11]:

from pathlib import Path
import re
import random

folder = Path.home() / "Documents" / "EngiLearn"
text_path = folder / "cleaned_text.txt"

text = text_path.read_text(encoding="utf-8")

# Extract readable lines from the notes
lines = []

for line in text.splitlines():
    line = re.sub(r"^PAGE \d+$", "", line).strip()
    line = re.sub(r"^[•\-–]\s*", "", line)

    if len(line.split()) >= 5:
        if not re.search(
            r"copyright|all rights reserved|creative commons|https?://",
            line,
            re.IGNORECASE
        ):
            lines.append(line)

# Remove duplicate lines while preserving order
lines = list(dict.fromkeys(lines))

# Select useful lines containing key concepts
keywords = [
    "deep learning", "neural network", "backpropagation",
    "gradient", "machine learning", "XOR", "training",
    "pytorch", "tensorflow"
]

study_notes = [
    line for line in lines
    if any(keyword in line.lower() for keyword in keywords)
]

print("STUDY NOTES\n")

for note in study_notes[:12]:
    print("-", note)

# Create simple practice questions from the extracted text
print("\nPRACTICE QUESTIONS\n")

for number, note in enumerate(study_notes[:5], start=1):
    print(f"{number}. Explain the following concept in your own words:")
    print(f"   {note}\n")

print("Note: These are practice prompts, not AI-generated questions.")


STUDY NOTES

- Lecture 1: Introduction to Deep Learning
- 1. Neural nets: A class of machine learning architectures that
- where parameterize parts of the program and let gradient-
- Breakthroughs in deep learning have been driven by a mixture of theory
- Theoretical grounding in important deep learning building blocks
- This is a deep learning class and you should try out all the latest AI assistants (they are
- pretty much all using deep learning). It's very important to play with them to learn what
- Existence proof for deep learning as a solution: The human brain?
- 1. Introduction to Deep Learning
- A brief history of Neural Networks
- PDP authors pointed to the backpropagation algorithm
- as a breakthrough, allowing multi-layer neural networks to be

PRACTICE QUESTIONS

1. Explain the following concept in your own words:
   Lecture 1: Introduction to Deep Learning

2. Explain the following concept in your own words:
   1. Neural nets: A class of machine learning architectures tha

In [12]:

from pathlib import Path
import json

folder = Path.home() / "Documents" / "EngiLearn"

# Concepts supported by the extracted lecture notes
study_material = {
    "Deep Learning": {
        "note": "Lecture 1 introduces deep learning and its role in machine learning.",
        "question": "What is the main topic of Lecture 1?",
        "options": [
            "Introduction to Deep Learning",
            "Database Management",
            "Computer Networks",
            "Web Development"
        ],
        "answer": "Introduction to Deep Learning"
    },
    "Neural Networks": {
        "note": "The lecture describes neural networks as machine-learning architectures that use stacks of linear transformations interleaved with pointwise nonlinearities.",
        "question": "According to the lecture, what do neural networks use?",
        "options": [
            "Only database tables",
            "Stacks of linear transformations and pointwise nonlinearities",
            "Only HTML pages",
            "Only computer networks"
        ],
        "answer": "Stacks of linear transformations and pointwise nonlinearities"
    },
    "Backpropagation": {
        "note": "The lecture explains that backpropagation enabled the training of multi-layer neural networks.",
        "question": "What breakthrough helped train multi-layer neural networks?",
        "options": [
            "Backpropagation",
            "Web scraping",
            "Database indexing",
            "File compression"
        ],
        "answer": "Backpropagation"
    }
}

# Display study notes
print("STUDY NOTES\n")

for concept, material in study_material.items():
    print(f"{concept}:")
    print(material["note"], "\n")

# Display quiz questions without revealing answers
print("QUIZ\n")

for number, (concept, material) in enumerate(
    study_material.items(), start=1
):
    print(f"{number}. {material['question']}")
    for option_number, option in enumerate(
        material["options"], start=1
    ):
        print(f"   {option_number}. {option}")
    print()

# Save material for the next stage
output_path = folder / "learning_material.json"

output_path.write_text(
    json.dumps(study_material, indent=4),
    encoding="utf-8"
)

print("Learning material saved to:", output_path)


STUDY NOTES

Deep Learning:
Lecture 1 introduces deep learning and its role in machine learning. 

Neural Networks:
The lecture describes neural networks as machine-learning architectures that use stacks of linear transformations interleaved with pointwise nonlinearities. 

Backpropagation:
The lecture explains that backpropagation enabled the training of multi-layer neural networks. 

QUIZ

1. What is the main topic of Lecture 1?
   1. Introduction to Deep Learning
   2. Database Management
   3. Computer Networks
   4. Web Development

2. According to the lecture, what do neural networks use?
   1. Only database tables
   2. Stacks of linear transformations and pointwise nonlinearities
   3. Only HTML pages
   4. Only computer networks

3. What breakthrough helped train multi-layer neural networks?
   1. Backpropagation
   2. Web scraping
   3. Database indexing
   4. File compression

Learning material saved to: C:\Users\Chetana\Documents\EngiLearn\learning_material.json


In [13]:
%pip install psutil


In [14]:

import psutil
import shutil
import subprocess
from pathlib import Path

# RAM
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
available_ram = psutil.virtual_memory().available / (1024 ** 3)

# Free storage on the drive containing your Documents folder
drive = Path.home().anchor
free_gb = shutil.disk_usage(drive).free / (1024 ** 3)

print(f"Total RAM: {ram_gb:.1f} GB")
print(f"Currently available RAM: {available_ram:.1f} GB")
print(f"Free disk space: {free_gb:.1f} GB")

# Graphics card
result = subprocess.run(
    [
        "powershell", "-NoProfile", "-Command",
        "Get-CimInstance Win32_VideoController | "
        "Select-Object -ExpandProperty Name"
    ],
    capture_output=True,
    text=True
)

print("\nGraphics card(s):")
print(result.stdout.strip() or "Could not detect graphics card")


Total RAM: 3.7 GB
Currently available RAM: 0.3 GB
Free disk space: 58.4 GB

Graphics card(s):
Intel(R) UHD Graphics
